In [1]:
# # XGBoost for regression, by hand
# We grow XGBoost's regression tree from scratch with NumPy on four students (CGPA and package in LPA),
# check every similarity score, gain and output value from the Note, then compare with the real XGBoost library
# and with scikit-learn.

In [2]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import GradientBoostingRegressor

## 1. The data and the first model (the mean)

In [3]:
# four students: CGPA (input) and package in LPA (output)
df = pd.DataFrame({"cgpa": [6.7, 9.0, 7.5, 5.0], "package": [4.5, 11.0, 6.0, 8.0]})
x, y = df["cgpa"].to_numpy(), df["package"].to_numpy()

# stage 1: the base model predicts the mean for everyone
f0 = y.mean()
df["pred1"] = f0
# residuals: actual minus predicted
df["res1"] = y - f0
print("mean:", f0)
df

mean: 7.375


,cgpa,package,pred1,res1
0,6.7,4.5,7.375,-2.875
1,9.0,11.0,7.375,3.625
2,7.5,6.0,7.375,-1.375
3,5.0,8.0,7.375,0.625


## 2. Similarity score, gain and output value

In [4]:
def similarity(r, lam=0.0):
    # (sum of residuals)^2 / (number of residuals + lambda)
    return r.sum() ** 2 / (len(r) + lam)

def output(r, lam=0.0):
    # sum of residuals / (number of residuals + lambda)
    return r.sum() / (len(r) + lam)

r = df["res1"].to_numpy()
print("root similarity:", similarity(r))

# candidate thresholds: midpoints of the sorted CGPA values
xs = np.sort(x)
thresholds = (xs[1:] + xs[:-1]) / 2
for t in thresholds:
    left, right = r[x < t], r[x >= t]
    gain = similarity(left) + similarity(right) - similarity(r)
    print(f"CGPA < {t:.2f}: S_L={similarity(left):.4f}  S_R={similarity(right):.4f}  gain={gain:.4f}")

root similarity: 0.0
CGPA < 5.85: S_L=0.3906  S_R=0.1302  gain=0.5208
CGPA < 7.10: S_L=2.5312  S_R=2.5312  gain=5.0625
CGPA < 8.25: S_L=4.3802  S_R=13.1406  gain=17.5208


## 3. A from-scratch XGBoost regression tree (exact greedy, lambda, gamma, max_depth)

In [5]:
def grow(x, r, lam=0.0, gamma=0.0, depth=0, max_depth=2):
    """Return a nested dict: a leaf {'out': w} or a split {'t', 'gain', 'left', 'right'}."""
    node = {"res": r, "sim": similarity(r, lam), "out": output(r, lam)}
    if depth == max_depth or len(r) < 2:
        return node
    # try every midpoint and keep the largest gain
    xs = np.unique(x)
    best = None
    for t in (xs[1:] + xs[:-1]) / 2:
        m = x < t
        gain = similarity(r[m], lam) + similarity(r[~m], lam) - similarity(r, lam)
        if best is None or gain > best[1]:
            best = (t, gain)
    t, gain = best
    m = x < t
    node.update(t=t, gain=gain,
                left=grow(x[m], r[m], lam, gamma, depth + 1, max_depth),
                right=grow(x[~m], r[~m], lam, gamma, depth + 1, max_depth))
    # pruning, bottom up: a split whose children are leaves goes if gain - gamma < 0
    if "t" not in node["left"] and "t" not in node["right"] and gain - gamma < 0:
        for k in ("t", "gain", "left", "right"):
            node.pop(k)
    return node

def show(node, indent=""):
    if "t" in node:
        print(f"{indent}CGPA < {node['t']:.2f}  (similarity {node['sim']:.2f}, gain {node['gain']:.2f})")
        show(node["left"], indent + "    ")
        show(node["right"], indent + "    ")
    else:
        print(f"{indent}leaf: residuals {np.round(node['res'], 3)}, similarity {node['sim']:.2f}, output {node['out']:.4f}")

def predict(node, v):
    while "t" in node:
        node = node["left"] if v < node["t"] else node["right"]
    return node["out"]

tree1 = grow(x, r)
show(tree1)

CGPA < 8.25  (similarity 0.00, gain 17.52)
    CGPA < 5.85  (similarity 4.38, gain 5.04)
        leaf: residuals [0.625], similarity 0.39, output 0.6250
        leaf: residuals [-2.875 -1.375], similarity 9.03, output -2.1250
    leaf: residuals [3.625], similarity 13.14, output 3.6250


## 4. Stage 2: mean + eta x tree output

In [6]:
eta = 0.3
df["pred2"] = f0 + eta * np.array([predict(tree1, v) for v in x])
df["res2"] = y - df["pred2"]
df.round(4)

,cgpa,package,pred1,res1,pred2,res2
0,6.7,4.5,7.375,-2.875,6.7375,-2.2375
1,9.0,11.0,7.375,3.625,8.4625,2.5375
2,7.5,6.0,7.375,-1.375,6.7375,-0.7375
3,5.0,8.0,7.375,0.625,7.5625,0.4375


## 5. Check: scikit-learn gives the same stage-2 predictions
With lambda = 0 the gain is the drop in squared error, so the XGBoost tree equals a squared-error regression tree
grown on the residuals; GradientBoostingRegressor starts from the mean and adds learning_rate x tree.

In [7]:
gbr = GradientBoostingRegressor(n_estimators=1, learning_rate=0.3, max_depth=2).fit(x.reshape(-1, 1), y)
print("sklearn thresholds:", gbr.estimators_[0, 0].tree_.threshold[gbr.estimators_[0, 0].tree_.feature >= 0])
print("sklearn:", gbr.predict(x.reshape(-1, 1)))
print("ours:   ", df["pred2"].to_numpy())
assert np.allclose(gbr.predict(x.reshape(-1, 1)), df["pred2"])

sklearn thresholds: [8.25      5.8499999]
sklearn: [6.7375 8.4625 6.7375 7.5625]
ours:    [6.7375 8.4625 6.7375 7.5625]


## 6. Check: the real XGBoost library
We set every setting to match the hand calculation. The dump shows each split's gain and each leaf's value;
XGBoost stores leaf values already multiplied by eta (0.3 x 0.625 = 0.1875).

In [8]:
X = x.reshape(-1, 1)
for lam, gamma in [(0, 0), (1, 0), (0, 6)]:
    model = xgb.XGBRegressor(n_estimators=1, max_depth=2, learning_rate=0.3, reg_lambda=lam, gamma=gamma,
                             min_child_weight=0, base_score=f0, tree_method="exact", device="cpu",
                             random_state=42).fit(X, y)
    print(f"lambda={lam}, gamma={gamma}: predictions {model.predict(X)}")
    print(model.get_booster().get_dump(with_stats=True)[0])
# the lambda = 0 predictions are our stage 2
model = xgb.XGBRegressor(n_estimators=1, max_depth=2, learning_rate=0.3, reg_lambda=0, min_child_weight=0,
                         base_score=f0, tree_method="exact", device="cpu").fit(X, y)
assert np.allclose(model.predict(X), df["pred2"], atol=1e-5)

lambda=0, gamma=0: predictions [6.7375 8.4625 6.7375 7.5625]
0:[f0<8.25] yes=1,no=2,missing=1,gain=17.520834,cover=4
	1:[f0<5.8499999] yes=3,no=4,missing=3,gain=5.04166651,cover=3
		3:leaf=0.1875,cover=1
		4:leaf=-0.637500048,cover=2
	2:leaf=1.0875001,cover=1

lambda=1, gamma=0: predictions [6.95    7.91875 6.95    7.46875]
0:[f0<8.25] yes=1,no=2,missing=1,gain=9.85546875,cover=4
	1:[f0<5.8499999] yes=3,no=4,missing=3,gain=2.93098927,cover=3
		3:leaf=0.09375,cover=1
		4:leaf=-0.425000012,cover=2
	2:leaf=0.543750048,cover=1

lambda=0, gamma=6: predictions [7.0125 8.4625 7.0125 7.0125]
0:[f0<8.25] yes=1,no=2,missing=1,gain=17.520834,cover=4
	1:leaf=-0.362500012,cover=3
	2:leaf=1.0875001,cover=1



## 7. The second tree, on the stage-2 residuals

In [9]:
tree2 = grow(x, df["res2"].to_numpy())
show(tree2)
df["pred3"] = df["pred2"] + eta * np.array([predict(tree2, v) for v in x])
df["res3"] = y - df["pred3"]
df.round(4)

CGPA < 8.25  (similarity 0.00, gain 8.59)
    CGPA < 5.85  (similarity 2.15, gain 2.47)
        leaf: residuals [0.438], similarity 0.19, output 0.4375
        leaf: residuals [-2.238 -0.737], similarity 4.43, output -1.4875
    leaf: residuals [2.537], similarity 6.44, output 2.5375


,cgpa,package,pred1,res1,pred2,res2,pred3,res3
0,6.7,4.5,7.375,-2.875,6.7375,-2.2375,6.2912,-1.7912
1,9.0,11.0,7.375,3.625,8.4625,2.5375,9.2238,1.7762
2,7.5,6.0,7.375,-1.375,6.7375,-0.7375,6.2912,-0.2912
3,5.0,8.0,7.375,0.625,7.5625,0.4375,7.6938,0.3063


## 8. Extra: lambda shrinks similarity scores and outputs

In [10]:
for lam in (0, 1):
    print(f"lambda = {lam}")
    show(grow(x, r, lam=lam))

lambda = 0
CGPA < 8.25  (similarity 0.00, gain 17.52)
    CGPA < 5.85  (similarity 4.38, gain 5.04)
        leaf: residuals [0.625], similarity 0.39, output 0.6250
        leaf: residuals [-2.875 -1.375], similarity 9.03, output -2.1250
    leaf: residuals [3.625], similarity 13.14, output 3.6250
lambda = 1
CGPA < 8.25  (similarity 0.00, gain 9.86)
    CGPA < 5.85  (similarity 3.29, gain 2.93)
        leaf: residuals [0.625], similarity 0.20, output 0.3125
        leaf: residuals [-2.875 -1.375], similarity 6.02, output -1.4167
    leaf: residuals [3.625], similarity 6.57, output 1.8125


## 9. Extra: gamma prunes weak splits

In [11]:
for gamma in (0, 6, 20):
    print(f"gamma = {gamma}")
    show(grow(x, r, gamma=gamma))

gamma = 0
CGPA < 8.25  (similarity 0.00, gain 17.52)
    CGPA < 5.85  (similarity 4.38, gain 5.04)
        leaf: residuals [0.625], similarity 0.39, output 0.6250
        leaf: residuals [-2.875 -1.375], similarity 9.03, output -2.1250
    leaf: residuals [3.625], similarity 13.14, output 3.6250
gamma = 6
CGPA < 8.25  (similarity 0.00, gain 17.52)
    leaf: residuals [-2.875 -1.375  0.625], similarity 4.38, output -1.2083
    leaf: residuals [3.625], similarity 13.14, output 3.6250
gamma = 20
leaf: residuals [-2.875  3.625 -1.375  0.625], similarity 0.00, output 0.0000


In [12]:
# two trees in XGBoost match our stage 3
model2 = xgb.XGBRegressor(n_estimators=2, max_depth=2, learning_rate=0.3, reg_lambda=0, min_child_weight=0,
                          base_score=f0, tree_method="exact", device="cpu").fit(X, y)
print(model2.predict(X), df["pred3"].to_numpy())
assert np.allclose(model2.predict(X), df["pred3"], atol=1e-5)

[6.29125 9.22375 6.29125 7.69375] [6.29125 9.22375 6.29125 7.69375]


## 10. Check: a parent with a small gain stays if its child split is kept
Residuals 1, -1, -1, 1: the root gain (1.33) is below gamma = 2, the child gain (2.67) is above it. Bottom-up pruning keeps both; gamma = 3 removes both.

In [13]:
Xp = np.array([[1.0], [2.0], [3.0], [4.0]])
rp = np.array([1.0, -1.0, -1.0, 1.0])
for gamma in (0, 2, 3):
    m = xgb.XGBRegressor(n_estimators=1, max_depth=2, learning_rate=1.0, reg_lambda=0, gamma=gamma, min_child_weight=0,
                         base_score=0, tree_method="exact", device="cpu").fit(Xp, rp)
    print(f"gamma={gamma}:")
    print(m.get_booster().get_dump(with_stats=True)[0])

gamma=0:
0:[f0<3.5] yes=1,no=2,missing=1,gain=1.33333325,cover=4
	1:[f0<1.5] yes=3,no=4,missing=3,gain=2.66666675,cover=3
		3:leaf=1,cover=1
		4:leaf=-1,cover=2
	2:leaf=1,cover=1

gamma=2:
0:[f0<3.5] yes=1,no=2,missing=1,gain=1.33333325,cover=4
	1:[f0<1.5] yes=3,no=4,missing=3,gain=2.66666675,cover=3
		3:leaf=1,cover=1
		4:leaf=-1,cover=2
	2:leaf=1,cover=1



gamma=3:
0:leaf=-0,cover=4

